# t-SNE — Hands-On Exercises

## Scenario: Who are the customers behind the receipts?

A small chain of outlets has behavioural features for each customer: monthly
spend, visit frequency, basket size, promo sensitivity, online share, and so
on. The marketing team suspects there are distinct **customer segments**, but
nobody has labelled them.

In this notebook you will:

1. generate a reproducible synthetic outlet/customer dataset with known groups,
2. calibrate Gaussian bandwidths so that each point matches a target perplexity,
3. build symmetric joint probabilities and Student-t similarities,
4. check the analytic KL gradient against finite differences,
5. run from-scratch t-SNE and measure neighborhood preservation,
6. observe how perplexity and early exaggeration change the embedding.

Every cell runs top-to-bottom in a few seconds. Cells marked **TODO** contain a
small challenge; change the values and watch what happens.

## Setup

The notebook works whether you launch Jupyter from the repository root or from
this module folder.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

cwd = Path.cwd()
for candidate in (cwd, cwd / "02_unsupervised_learning" / "05_tsne"):
    if (candidate / "implementation.py").exists():
        sys.path.insert(0, str(candidate))
        break

from implementation import (
    FEATURE_NAMES,
    SEGMENT_NAMES,
    TSNE,
    compute_pairwise_affinities,
    generate_customer_segments,
    joint_probabilities,
    kl_divergence,
    kl_gradient,
    knn_preservation,
    pairwise_squared_distances,
    q_distribution,
    trustworthiness,
)

np.set_printoptions(precision=4, suppress=True)


def row_entropies(P):
    safe = np.where(P > 0.0, P, 1.0)
    return -np.sum(np.where(P > 0.0, P * np.log(safe), 0.0), axis=1)


X, labels = generate_customer_segments(
    n_samples=120, n_features=10, n_clusters=4, random_state=42
)
print(f"Feature matrix: {X.shape}")
print(f"Feature names : {FEATURE_NAMES[:6]} ...")

## Exercise 1 — Inspect the synthetic customer segments

The generator creates balanced groups around different behavioural profiles.

**TODO:** Generate a new dataset with `n_clusters=5` and `random_state=7`.
Which segment sizes do you get?

**Hint:** `np.bincount(labels)` returns the number of rows per segment.

In [ ]:
print(f"Samples : {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print("\nSegment sizes:")
for index, count in enumerate(np.bincount(labels)):
    print(f"  {SEGMENT_NAMES[index]:22s}: {count}")

print("\nFirst two rows:")
print(X[:2])

## Exercise 2 — Perplexity is a bandwidth calibration

For every point $i$, t-SNE binary-searches a precision $\beta_i$ so that the
conditional Gaussian

$$p_{j|i} = \frac{\exp(-\beta_i \lVert x_i - x_j \rVert^2)}
{\sum_{k \neq i} \exp(-\beta_i \lVert x_i - x_k \rVert^2)}$$

has entropy $H(P_i) = \log(\text{perplexity})$.

**TODO:** Try perplexities `5`, `15`, and `30`. How do `sigma` and entropy
change?

**Hint:** A larger perplexity needs a wider Gaussian, so `sigma` should grow.

In [ ]:
perplexity = 15.0
P_conditional, sigmas = compute_pairwise_affinities(X, perplexity=perplexity)

entropies = row_entropies(P_conditional)
print(f"Target log(perplexity): {np.log(perplexity):.6f}")
print(f"Mean row entropy      : {entropies.mean():.6f}")
print(f"Max entropy error     : {np.max(np.abs(entropies - np.log(perplexity))):.2e}")
print(f"sigma range           : {sigmas.min():.3f} - {sigmas.max():.3f}")
print(f"Rows sum to 1?        : {np.allclose(P_conditional.sum(axis=1), 1.0)}")

## Exercise 3 — Symmetric P and Student-t Q

The conditional matrix is symmetrized and normalized:

$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}.$$

In the embedding, t-SNE uses a heavy-tailed Student-t kernel:

$$q_{ij} = \frac{(1 + \lVert y_i - y_j \rVert^2)^{-1}}
{\sum_{k \neq l} (1 + \lVert y_k - y_l \rVert^2)^{-1}}.$$

**TODO:** Replace the random `Y` with a deliberately stretched embedding
(for example `Y * 5`). Does `Q` become more concentrated?

**Hint:** Compare `Q.max()` and `np.count_nonzero(Q > 1e-3)`.

In [ ]:
P = joint_probabilities(P_conditional)
rng = np.random.default_rng(0)
Y_random = rng.normal(size=(X.shape[0], 2)) * 0.2
Q = q_distribution(Y_random)

print(f"P symmetric? {np.allclose(P, P.T)}")
print(f"P sums to 1? {np.isclose(P.sum(), 1.0)}")
print(f"Q symmetric? {np.allclose(Q, Q.T)}")
print(f"Q sums to 1? {np.isclose(Q.sum(), 1.0)}")
print(f"KL(P || Q) : {kl_divergence(P, Q):.4f}")

## Exercise 4 — Check the analytic gradient

The gradient used by gradient descent is

$$\frac{\partial \mathrm{KL}}{\partial y_i}
= 4 \sum_j (p_{ij} - q_{ij})
\frac{y_i - y_j}{1 + \lVert y_i - y_j \rVert^2}.$$

A finite-difference estimate should match it closely.

**TODO:** Change `epsilon` to `1e-4` and `1e-7`. Which value gives the
smallest error?

**Hint:** Too large an epsilon truncates the derivative; too small a value
suffers from floating-point cancellation.

In [ ]:
small_X = X[:8]
P_small_cond, _ = compute_pairwise_affinities(small_X, perplexity=4.0)
P_small = joint_probabilities(P_small_cond)
Y_small = rng.normal(size=(8, 2)) * 0.3

Q_small = q_distribution(Y_small)
analytic = kl_gradient(P_small, Q_small, Y_small)

epsilon = 1e-6
numerical = np.zeros_like(Y_small)
for i in range(Y_small.shape[0]):
    for d in range(2):
        plus = Y_small.copy()
        minus = Y_small.copy()
        plus[i, d] += epsilon
        minus[i, d] -= epsilon
        numerical[i, d] = (
            kl_divergence(P_small, q_distribution(plus))
            - kl_divergence(P_small, q_distribution(minus))
        ) / (2.0 * epsilon)

print(f"Max absolute gradient error: {np.max(np.abs(analytic - numerical)):.2e}")
print("Analytic gradient (first row) :", analytic[0])
print("Numerical gradient (first row):", numerical[0])

## Exercise 5 — Run from-scratch t-SNE

The optimizer uses early exaggeration (multiply `P` by 12 for the first
iterations), momentum, and adaptive gains.

**TODO:** Increase `n_iter` to `600`. Does trustworthiness improve, and how
much longer does it take?

**Hint:** `model.kl_history_` records the raw KL divergence at every iteration.

In [ ]:
start = time.time()
model = TSNE(
    perplexity=15.0,
    learning_rate=200.0,
    n_iter=300,
    early_exaggeration=12.0,
    early_exaggeration_iter=100,
    random_state=42,
)
embedding = model.fit_transform(X)
elapsed = time.time() - start

print(f"Embedding shape : {embedding.shape}")
print(f"Runtime         : {elapsed:.2f} s")
print(f"KL first        : {model.kl_history_[0]:.4f}")
print(f"KL final        : {model.kl_history_[-1]:.4f}")
print(f"Trustworthiness : {trustworthiness(X, embedding, 10):.3f}")
print(f"kNN preservation: {knn_preservation(X, embedding, 10):.3f}")

## Exercise 6 — How does perplexity change the picture?

Low perplexity focuses on very local neighborhoods; high perplexity keeps more
global relationships but can blur small groups. Use trustworthiness and kNN
preservation as quantitative checks.

**TODO:** Add `perplexity=60` to the list. Is it better or worse than `30` on
this dataset?

**Hint:** With only 120 samples, a very high perplexity approaches the whole
dataset and may wash out segment structure.

In [ ]:
rows = []
for perplexity in (5.0, 15.0, 30.0):
    candidate = TSNE(
        perplexity=perplexity,
        n_iter=250,
        early_exaggeration_iter=90,
        random_state=42,
    )
    candidate_embedding = candidate.fit_transform(X)
    rows.append(
        {
            "perplexity": perplexity,
            "final_kl": candidate.kl_history_[-1],
            "trustworthiness": trustworthiness(X, candidate_embedding, 10),
            "knn_preservation": knn_preservation(X, candidate_embedding, 10),
        }
    )

print(f"{'perplexity':>10} {'final KL':>10} {'trust':>8} {'kNN':>8}")
for row in rows:
    print(
        f"{row['perplexity']:>10.0f} {row['final_kl']:>10.4f} "
        f"{row['trustworthiness']:>8.3f} {row['knn_preservation']:>8.3f}"
    )

## Exercise 7 — Why early exaggeration?

Early exaggeration multiplies the attractive forces for the first iterations.
This helps clusters tighten and separate before the optimizer refines their
internal layout. On very easy, well-separated data the difference can be
small, and the exaggerated phase is not always better for the final KL; on
harder or overlapping data it usually helps separate the groups.

**TODO:** Set `early_exaggeration=1.0` and compare the final KL and
trustworthiness with the default `12.0`.

**Hint:** Keep `early_exaggeration_iter` at `90` so that only the multiplier
changes.

In [ ]:
for exaggeration in (1.0, 12.0):
    candidate = TSNE(
        perplexity=15.0,
        n_iter=250,
        early_exaggeration=exaggeration,
        early_exaggeration_iter=90,
        random_state=42,
    )
    candidate_embedding = candidate.fit_transform(X)
    print(
        f"early_exaggeration={exaggeration:4.1f}  "
        f"final KL={candidate.kl_history_[-1]:.4f}  "
        f"trust={trustworthiness(X, candidate_embedding, 10):.3f}"
    )

## Summary

You have practiced the full t-SNE workflow:

- ✅ generated reproducible outlet/customer data with known segments,
- ✅ calibrated per-point Gaussian bandwidths to a target perplexity,
- ✅ built symmetric joint probabilities and Student-t similarities,
- ✅ verified the exact KL gradient with finite differences,
- ✅ trained t-SNE with early exaggeration, momentum and gains,
- ✅ measured trustworthiness and kNN preservation,
- ✅ explored the perplexity and early-exaggeration trade-offs.

### Next steps

1. Read `README.md` for the full derivation and the pitfalls of reading cluster
   sizes or inter-cluster distances.
2. Run `sklearn_example.py` to compare with scikit-learn and PCA.
3. Try a real dataset and remember: t-SNE is exploratory visualization, not a
   clustering algorithm by itself.